# 05 - Análise Avançada com Dados de Mercado

Este notebook demonstra análises avançadas para detecção de fraude:

1. **Phantom Assets Detection** - Identifica ativos fictícios
2. **Peer Comparison** - Compara fundos REAG com mercado
3. **Concentration Analysis** - Detecta concentração excessiva
4. **Market Price Validation** - Valida preços declarados
5. **Integrated Analysis** - Visão consolidada

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Importar analyzers avançados
from src.analyzers.phantom_assets import PhantomAssetDetector
from src.analyzers.peer_comparison import PeerComparisonAnalyzer
from src.analyzers.concentration import ConcentrationAnalyzer
from src.analyzers.market_data import MarketDataValidator

# Processadores existentes
from src.processors.data_processor import DataProcessor
from config.settings import Config

# Configuração de visualização
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Módulos importados com sucesso!")

## Configuração e Carregamento de Dados

In [ ]:
config = Config()
processor = DataProcessor(config)

# Diretórios
REPORTS_DIR = config.REPORTS_DIR / 'advanced_analysis'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Diretório de relatórios: {REPORTS_DIR}")

In [ ]:
# Carregar lista de fundos REAG (do notebook 02)
reag_funds_path = config.PROCESSED_DATA_DIR / 'reag_fund_list.csv'

if reag_funds_path.exists():
    df_reag_funds = pd.read_csv(reag_funds_path)
    reag_cnpjs = df_reag_funds['CNPJ_FUNDO'].tolist()
    print(f"✅ {len(reag_cnpjs)} fundos REAG carregados")
else:
    print("⚠️  Execute primeiro o notebook 02_identify_reag_funds.ipynb")
    reag_cnpjs = []  # Lista vazia para demonstração

In [ ]:
# Carregar Cadastro
cadastro_path = config.RAW_DATA_DIR / 'cad_fi.csv'

if cadastro_path.exists():
    df_cadastro = pd.read_csv(cadastro_path, encoding='latin1', sep=';')
    print(f"✅ Cadastro carregado: {len(df_cadastro):,} fundos")
else:
    print("⚠️  Cadastro não encontrado. Execute notebook 01_data_collection.ipynb")
    df_cadastro = None

In [ ]:
# Carregar Informe Diário processado (do notebook 03)
informe_path = config.PROCESSED_DATA_DIR / 'reag_informe_diario_processed.csv'

if informe_path.exists():
    df_informe = pd.read_csv(informe_path, sep=';', parse_dates=['DT_COMPTC'])
    print(f"✅ Informe Diário carregado: {len(df_informe):,} registros")
else:
    print("⚠️  Execute notebook 03_flow_analysis.ipynb para processar dados")
    df_informe = None

In [ ]:
# Carregar CDA (Composição de Carteira)
# Exemplo: carregar um mês específico
cda_path = config.RAW_DATA_DIR / 'cda_fi_202401.csv'

if cda_path.exists():
    df_cda = pd.read_csv(cda_path, encoding='latin1', sep=';')
    # Filtrar apenas fundos REAG
    if reag_cnpjs:
        df_cda_reag = df_cda[df_cda['CNPJ_FUNDO'].isin(reag_cnpjs)]
        print(f"✅ CDA carregado: {len(df_cda_reag):,} posições de fundos REAG")
    else:
        df_cda_reag = df_cda
        print(f"✅ CDA carregado: {len(df_cda):,} posições (todos os fundos)")
else:
    print("⚠️  CDA não encontrado. Execute notebook 01_data_collection.ipynb")
    df_cda_reag = None

## 1️⃣ Phantom Assets Detection (Ativos Fictícios)

Identifica ativos que NÃO EXISTEM em registros oficiais.

**Red Flag Crítico:** Ativos fantasma = fraude direta.

In [ ]:
# Inicializar detector
phantom_detector = PhantomAssetDetector()

# Atualizar registros de ativos válidos
phantom_detector.update_registries()

# Carregar fundos do cadastro
if df_cadastro is not None:
    phantom_detector.load_funds_from_cadastro(cadastro_path)

In [ ]:
# Detectar ativos fantasma
if df_cda_reag is not None:
    phantom_report = phantom_detector.generate_phantom_report(
        df_cda_reag,
        output_path=REPORTS_DIR / 'phantom_assets.csv'
    )
    
    # Visualizar
    if not phantom_report.empty:
        display(phantom_report.head(10))
else:
    print("⚠️  Dados de CDA não disponíveis")

## 2️⃣ Peer Comparison Analysis

Compara fundos REAG com fundos similares do mercado.

**Red Flags:**
- Retornos muito acima de peers (Z-score > 3)
- Volatilidade suspeitsamente baixa
- Sharpe ratio "bom demais para ser verdade"

In [ ]:
# Inicializar analyzer
peer_analyzer = PeerComparisonAnalyzer()

# Carregar categorias de fundos
if df_cadastro is not None:
    peer_analyzer.load_fund_categories(df_cadastro)

In [ ]:
# Gerar relatório de peer comparison
if df_informe is not None and reag_cnpjs:
    peer_reports = peer_analyzer.generate_peer_report(
        target_funds=reag_cnpjs,
        informe_df=df_informe,
        output_path=REPORTS_DIR
    )
    
    # Visualizar outliers
    if not peer_reports['peer_comparison'].empty:
        outliers = peer_reports['peer_comparison'][peer_reports['peer_comparison']['is_outlier']]
        if not outliers.empty:
            print("\n⚠️  OUTLIERS DETECTADOS:")
            display(outliers)
else:
    print("⚠️  Dados de Informe Diário não disponíveis")

In [ ]:
# Visualização: Z-scores de retorno
if df_informe is not None and 'peer_reports' in locals():
    peer_comp = peer_reports['peer_comparison']
    
    if not peer_comp.empty:
        plt.figure(figsize=(12, 6))
        
        # Scatter plot: Z-score retorno vs Sharpe
        colors = peer_comp['is_outlier'].map({True: 'red', False: 'blue'})
        
        plt.scatter(peer_comp['return_zscore'], 
                   peer_comp['sharpe_zscore'],
                   c=colors, s=100, alpha=0.6)
        
        # Linhas de threshold (Z = ±3)
        plt.axhline(y=3, color='red', linestyle='--', alpha=0.5, label='Threshold (Z=3)')
        plt.axhline(y=-3, color='red', linestyle='--', alpha=0.5)
        plt.axvline(x=3, color='red', linestyle='--', alpha=0.5)
        plt.axvline(x=-3, color='red', linestyle='--', alpha=0.5)
        
        plt.xlabel('Z-Score Retorno vs Peers')
        plt.ylabel('Z-Score Sharpe vs Peers')
        plt.title('Fundos REAG vs Peers - Outlier Detection')
        plt.legend(['Threshold', 'Outlier', 'Normal'])
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## 3️⃣ Concentration Analysis

Detecta concentração excessiva em poucos ativos.

**Red Flags:**
- HHI > 0.25 (muito concentrado)
- Violação de limites regulatórios
- Concentração em partes relacionadas

In [ ]:
# Inicializar analyzer
conc_analyzer = ConcentrationAnalyzer()

# Carregar categorias
if peer_analyzer.fund_categories:
    conc_analyzer.load_fund_categories(peer_analyzer.fund_categories)

In [ ]:
# Definir emissores relacionados (exemplo - ajustar conforme necessário)
# Em produção, carregar de base de dados ou pesquisa
related_issuers = {'REAG', 'CBSF', 'BANCO MASTER'}
conc_analyzer.set_related_issuers(related_issuers)

In [ ]:
# Gerar relatório de concentração
if df_cda_reag is not None:
    conc_reports = conc_analyzer.generate_concentration_report(
        df_cda_reag,
        target_funds=reag_cnpjs if reag_cnpjs else None,
        output_path=REPORTS_DIR
    )
    
    # Visualizar violações
    if not conc_reports['excessive_concentration'].empty:
        print("\n⚠️  CONCENTRAÇÃO EXCESSIVA:")
        display(conc_reports['excessive_concentration'].head(10))
else:
    print("⚠️  Dados de CDA não disponíveis")

In [ ]:
# Visualização: HHI vs Top 1 Position
if df_cda_reag is not None and 'conc_reports' in locals():
    excessive = conc_reports['excessive_concentration']
    
    if not excessive.empty:
        plt.figure(figsize=(12, 6))
        
        # Mapear severity para cores
        color_map = {'CRITICAL': 'red', 'HIGH': 'orange', 'MEDIUM': 'yellow'}
        colors = excessive['severity'].map(color_map)
        
        plt.scatter(excessive['hhi'], excessive['top1_pct'], 
                   c=colors, s=100, alpha=0.6)
        
        # Linhas de threshold
        plt.axhline(y=25, color='red', linestyle='--', alpha=0.5, label='Limite (25%)')
        plt.axvline(x=0.25, color='red', linestyle='--', alpha=0.5, label='HHI Crítico (0.25)')
        
        plt.xlabel('Índice Herfindahl (HHI)')
        plt.ylabel('Top 1 Position (%)')
        plt.title('Análise de Concentração - Fundos REAG')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## 4️⃣ Market Price Validation

Valida preços declarados vs preços de mercado.

**Red Flags:**
- Divergência > 10% (possível overvaluation)
- Padrão sistemático de sobrevalorização

**Nota:** Requer conexão com internet para Yahoo Finance.

In [ ]:
# Verificar se yfinance está instalado
try:
    import yfinance as yf
    print("✅ yfinance instalado")
    yfinance_available = True
except ImportError:
    print("⚠️  yfinance não instalado. Execute: pip install yfinance")
    yfinance_available = False

In [ ]:
# Inicializar validator
if yfinance_available:
    price_validator = MarketDataValidator()
    
    # Validar preços (sample pequeno para demonstração)
    if df_cda_reag is not None:
        print("\n⚠️  NOTA: Análise pode demorar dependendo do tamanho da amostra")
        print("         Usando amostra de 100 posições para demonstração\n")
        
        price_reports = price_validator.generate_price_report(
            df_cda_reag,
            sample_size=100,  # Limitar para demonstração
            output_path=REPORTS_DIR
        )
        
        # Visualizar manipulações
        if not price_reports['manipulation'].empty:
            print("\n🚨 MANIPULAÇÃO DE PREÇOS DETECTADA:")
            display(price_reports['manipulation'].head(10))
    else:
        print("⚠️  Dados de CDA não disponíveis")
else:
    print("⚠️  Pulando validação de preços (yfinance não instalado)")

## 5️⃣ Análise Integrada

Consolidação de todos os red flags encontrados.

In [ ]:
# Consolidar red flags por fundo
red_flags_summary = []

for cnpj in reag_cnpjs:
    flags = []
    severity = 'LOW'
    
    # Phantom assets
    if 'phantom_report' in locals() and not phantom_report.empty:
        phantom_funds = phantom_report['CNPJ_FUNDO'].unique() if 'CNPJ_FUNDO' in phantom_report.columns else []
        # Note: phantom assets are at asset level, not fund level in current implementation
        # Need to check if this fund holds any phantom assets
    
    # Peer outliers
    if 'peer_reports' in locals():
        outliers = peer_reports['peer_comparison']
        if cnpj in outliers[outliers['is_outlier']]['CNPJ_FUNDO'].values:
            flags.append('PEER_OUTLIER')
            severity = 'HIGH'
    
    # Concentration
    if 'conc_reports' in locals():
        excessive = conc_reports['excessive_concentration']
        if cnpj in excessive['CNPJ_FUNDO'].values:
            flags.append('EXCESSIVE_CONCENTRATION')
            if excessive[excessive['CNPJ_FUNDO'] == cnpj]['severity'].iloc[0] == 'CRITICAL':
                severity = 'CRITICAL'
    
    if flags:
        red_flags_summary.append({
            'CNPJ_FUNDO': cnpj,
            'red_flags': ', '.join(flags),
            'num_flags': len(flags),
            'severity': severity
        })

# Criar DataFrame
if red_flags_summary:
    df_red_flags = pd.DataFrame(red_flags_summary)
    df_red_flags = df_red_flags.sort_values('num_flags', ascending=False)
    
    print("\n" + "="*60)
    print("🚨 RESUMO DE RED FLAGS POR FUNDO")
    print("="*60)
    display(df_red_flags)
    
    # Salvar
    df_red_flags.to_csv(REPORTS_DIR / 'red_flags_summary.csv', index=False)
    print(f"\n💾 Resumo salvo em: {REPORTS_DIR / 'red_flags_summary.csv'}")
else:
    print("✅ Nenhum red flag consolidado encontrado")

## 📊 Dashboard Final

In [ ]:
print("\n" + "="*70)
print("📊 DASHBOARD DE ANÁLISE AVANÇADA - FUNDOS REAG")
print("="*70)

# Contadores
phantom_count = len(phantom_report) if 'phantom_report' in locals() and not phantom_report.empty else 0
outlier_count = peer_reports['peer_comparison']['is_outlier'].sum() if 'peer_reports' in locals() else 0
concentration_count = len(conc_reports['excessive_concentration']) if 'conc_reports' in locals() else 0
price_manip_count = len(price_reports['manipulation']) if 'price_reports' in locals() and yfinance_available else 0

print(f"\n🔍 ANÁLISES EXECUTADAS:")
print(f"   1. Phantom Assets: {phantom_count} ativos fictícios detectados")
print(f"   2. Peer Outliers: {outlier_count} fundos outliers")
print(f"   3. Concentration: {concentration_count} fundos com concentração excessiva")
print(f"   4. Price Manipulation: {price_manip_count} casos suspeitos")

total_issues = phantom_count + outlier_count + concentration_count + price_manip_count

print(f"\n🚨 TOTAL DE ISSUES: {total_issues}")

if total_issues > 0:
    print("\n⚠️  RED FLAGS DETECTADOS - Recomenda-se análise manual detalhada")
else:
    print("\n✅ Nenhum red flag crítico detectado nas análises automatizadas")

print(f"\n📁 Relatórios salvos em: {REPORTS_DIR}")
print("="*70)

## 📝 Próximos Passos

1. **Análise Manual:** Revisar fundos com múltiplos red flags
2. **Investigação Profunda:** Para casos críticos, coletar evidências adicionais
3. **Temporal Analysis:** Analisar evolução dos red flags ao longo do tempo
4. **Relatório Executivo:** Consolidar findings para stakeholders

### Análises Adicionais Possíveis:

- **Event Correlation:** Correlacionar red flags com notícias/sanções
- **Network Analysis:** Mapear relações entre fundos suspeitos
- **Liquidity Analysis:** Avaliar se resgates são compatíveis com liquidez
- **Credit Risk:** Avaliar qualidade de crédito de bonds em carteira